# Reusable Template: Simple Univariate & Bivariate Statistical Techniques

**Purpose.** This notebook is a ready-to-adapt template for the analyses covered in Chapter 4 of *Applied Univariate, Bivariate, and Multivariate Statistics Using Python* (Denis, 2021):

- Pearson product-moment correlation  
- Spearman’s rho  
- One-sample, independent-samples, and paired-samples *t*-tests  
- Binomial test  
- Chi-squared goodness-of-fit and contingency-table tests  

**How to use.**  
1. Replace the placeholder data-loading cell with your own data.  
2. Run the exploratory plots first—never skip them.  
3. Execute the formal tests only after you have visually inspected the relevant relationships.  
4. Fill in the interpretation markdown cells with substantive (not purely statistical) conclusions.  

**Core reminder from the chapter.** A correlation of zero does *not* mean “no relationship”; it means “no *linear* relationship.” Always plot. Statistical significance is not the same as scientific importance.


## 0. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, chisquare, binomtest  # or stats.binom_test in older scipy

# Reproducibility
np.random.seed(42)

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline


## 1. Data Loading & Quick Inspection

Replace the example below with your actual data source.  
Keep a short description of the variables and the research question in this cell.


In [ ]:
# --- REPLACE THIS BLOCK WITH YOUR DATA ---
# Example: synthetic continuous data for demonstration
n = 120
parent = np.random.normal(68, 2.5, n)
child  = 0.45 * parent + np.random.normal(30, 3.0, n)   # mild positive linear relationship

df = pd.DataFrame({
    "parent": parent,
    "child": child,
    "group": np.random.choice(["A", "B"], size=n, p=[0.5, 0.5]),
    "binary_outcome": np.random.binomial(1, 0.55, n)
})

# Quick look
print(df.head())
print("\nShape:", df.shape)
print("\nDescriptive statistics:")
print(df.describe().round(2))


## 2. Exploratory Data Analysis — Always Plot First

**Rule.** Never interpret a correlation (or any relationship coefficient) without a plot of the two variables.


In [ ]:
# Univariate distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["parent"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Parent distribution")
sns.histplot(df["child"], kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Child distribution")
plt.tight_layout()
plt.show()


In [ ]:
# Scatterplot / pairplot for continuous variables of interest
sns.pairplot(df[["parent", "child"]], diag_kind="hist", height=3.5)
plt.suptitle("Scatterplot matrix", y=1.02)
plt.show()


In [ ]:
# Boxplots by group (if applicable)
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="group", y="child", ax=ax)
ax.set_title("Child height by group")
plt.show()


## 3. Pearson Product-Moment Correlation

**What it measures.** Strength and direction of the *linear* relationship.  
**Formula reminder.** $r = \dfrac{\mathrm{cov}(x,y)}{s_x s_y}$ (standardized covariance).  
**Critical caveat.** $r \approx 0$ does **not** imply absence of a relationship—only absence of a linear one.


In [ ]:
# Pearson correlation + p-value
r, p = stats.pearsonr(df["parent"], df["child"])
print(f"Pearson r = {r:.4f}")
print(f"p-value   = {p:.4e}")
print(f"r² (variance explained) = {r**2:.3f}")
print(f"1 − r² (coefficient of humility) = {1 - r**2:.3f}")


**Interpretation notes (fill in):**  
- Direction and approximate strength:  
- Is the relationship visually linear?  
- Is the magnitude scientifically interesting given the variables?  
- What does the unexplained variance ($1-r^2$) remind us of?


## 4. Spearman’s Rho (Monotonic Relationships)

Use when the relationship is expected to be monotonic but not necessarily linear.  
Spearman is the Pearson correlation computed on the ranks.


In [ ]:
# Spearman correlation
rho, p_rho = stats.spearmanr(df["parent"], df["child"])
print(f"Spearman ρ = {rho:.4f}")
print(f"p-value    = {p_rho:.4e}")

# Side-by-side comparison
print("\nComparison:")
print(f"  Pearson r  = {r:.4f}")
print(f"  Spearman ρ = {rho:.4f}")
print("  If they differ substantially, inspect the scatter for non-linearity.")


## 5. One-Sample *t*-Test

**Null hypothesis example.** $H_0: \mu = \mu_0$ (e.g., population mean = 100 for IQ, or a historical benchmark).  
**Degrees of freedom.** $n-1$.


In [ ]:
# Example: test whether mean child height differs from a hypothesized value
mu0 = 65.0   # replace with a meaningful null value for your problem
t_stat, p_one = stats.ttest_1samp(df["child"], mu0)
print(f"One-sample t = {t_stat:.4f}")
print(f"p-value      = {p_one:.4f}")
print(f"Sample mean  = {df['child'].mean():.3f}")
print(f"Hypothesized μ₀ = {mu0}")


**Interpretation notes:**  
- Did we reject $H_0$?  
- What does failure to reject actually mean?  
- Is $\mu_0$ a scientifically interesting null, or merely a convenient number?


## 6. Independent-Samples *t*-Test

Assumptions (approximate): independence of observations, normality of residuals within groups, homogeneity of variance (Levene as a guide only).


In [ ]:
# Split by group
group_A = df.loc[df["group"] == "A", "child"]
group_B = df.loc[df["group"] == "B", "child"]

t_ind, p_ind = stats.ttest_ind(group_A, group_B, equal_var=True)  # set equal_var=False for Welch
print(f"Independent t = {t_ind:.4f}")
print(f"p-value       = {p_ind:.4f}")
print(f"Mean A = {group_A.mean():.3f}, Mean B = {group_B.mean():.3f}")

# Optional: Levene test for equality of variances
lev_stat, lev_p = stats.levene(group_A, group_B, center="mean")
print(f"\nLevene statistic = {lev_stat:.4f}, p = {lev_p:.4f}")
print("(Treat Levene p-value as a guide; it is itself sensitive to sample size.)")


## 7. Paired-Samples *t*-Test

Use when observations are naturally paired (before/after, matched subjects, repeated measures on the same unit).  
**Do not** treat paired data as independent.


In [ ]:
# Demo: create a paired example (e.g., trial 1 vs trial 2)
# In real work, load the actual paired columns
trial_1 = np.random.normal(10.3, 1.5, 30)
trial_2 = trial_1 - np.random.normal(1.1, 0.6, 30)   # systematic decrease

t_paired, p_paired = stats.ttest_rel(trial_1, trial_2)
print(f"Paired t = {t_paired:.4f}")
print(f"p-value  = {p_paired:.4e}")
print(f"Mean difference (trial1 − trial2) = {(trial_1 - trial_2).mean():.3f}")


## 8. Binomial Test

Suitable for binary, mutually exclusive outcomes with stationary success probability and independent trials.  
Classic example: number of heads in *n* flips of a coin hypothesized to be fair ($p=0.5$).


In [ ]:
# Example: observed successes out of n trials
successes = int(df["binary_outcome"].sum())
n_trials  = len(df)
p_null    = 0.50          # hypothesized probability under H0

# Modern scipy (>=1.7) uses binomtest; older versions used stats.binom_test
try:
    result = binomtest(successes, n=n_trials, p=p_null, alternative="two-sided")
    print(f"Observed successes = {successes} / {n_trials}")
    print(f"Binomial test p-value = {result.pvalue:.4f}")
except NameError:
    pval = stats.binom_test(successes, n=n_trials, p=p_null, alternative="two-sided")
    print(f"Observed successes = {successes} / {n_trials}")
    print(f"Binomial test p-value = {pval:.4f}")


## 9. Chi-Squared Goodness-of-Fit

Tests whether observed frequencies match a set of expected frequencies (often equal proportions or a fully specified multinomial).


In [ ]:
# Example: five categories with observed counts
observed = np.array([22, 18, 25, 15, 20])   # replace with your counts
# Expected under equal proportions
expected = np.full_like(observed, observed.sum() / len(observed), dtype=float)

chi2_stat, p_gof = chisquare(observed, f_exp=expected)
print(f"Chi-squared statistic = {chi2_stat:.4f}")
print(f"p-value               = {p_gof:.4f}")
print("Expected counts:", expected.round(2))


## 10. Chi-Squared Test of Independence (Contingency Table)

Null hypothesis: the row variable and the column variable are independent (no association) in the population.


In [ ]:
# Build a 2×2 (or larger) contingency table
# Example using the synthetic group and a binarized child variable
df["child_high"] = (df["child"] > df["child"].median()).astype(int)
contingency = pd.crosstab(df["group"], df["child_high"])
print("Observed contingency table:")
print(contingency)

chi2, p_cont, dof, expected = chi2_contingency(contingency)
print(f"\nChi-squared = {chi2:.4f}")
print(f"p-value     = {p_cont:.4f}")
print(f"Degrees of freedom = {dof}")
print("\nExpected frequencies under independence:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(2))


## 11. Synthesis & Critical Reflection

Use this section to write a short, honest summary.

**What did we learn?**  
-  

**Which results are scientifically meaningful versus merely statistically detectable?**  
-  

**Where did the data or the design limit our conclusions?**  
-  

**What would be the next natural analysis (e.g., regression, ANOVA, non-parametric alternatives)?**  
-  

---
*Remember the chapter’s central admonition: simple tests are often the most misused. Pair every coefficient with a plot, every p-value with a judgment of practical importance, and every rejection of H₀ with a clear statement of what the null actually claimed.*
